# 04 · Tune and select a model
Run `RandomizedSearchCV` with eight candidate settings and five stratified group folds per algorithm. Select the winning algorithm by mean validation macro F1, before opening the held-out test set.

In [1]:
from pathlib import Path
import sys

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

import pandas as pd
from IPython.display import display
from sklearn.model_selection import RandomizedSearchCV
from src.train import SEARCH_SPACE, model_candidates
from src.preprocessing import METRICS, REPORTS, SEED, five_folds, make_pipeline, make_split, write_json

X_train, _, y_train, _ = make_split()
settings = {}
folds = five_folds(X_train, y_train)
rows = []
for name, model in model_candidates().items():
    space = {f"model__{key}": values for key, values in SEARCH_SPACE[name].items()}
    search = RandomizedSearchCV(
        make_pipeline(model), space, n_iter=8, scoring="f1_macro",
        cv=folds, random_state=SEED, n_jobs=1,
        refit=False, error_score="raise",
    )
    search.fit(X_train, y_train)
    settings[name] = search.best_params_
    rows.append({
        "model": name,
        "cv_macro_f1": search.best_score_,
        "cv_macro_f1_std": search.cv_results_["std_test_score"][search.best_index_],
    })
    print(f"{name}: {search.best_params_}")

tuned = pd.DataFrame(rows).sort_values("cv_macro_f1", ascending=False, kind="stable")
winner = tuned.iloc[0]["model"]
write_json(REPORTS / "tuned_settings.json", {"winner": winner, "models": settings})
tuned.to_csv(METRICS / "tuned_cv.csv", index=False)
display(tuned.round(4))
print("Selected by training CV:", winner)

Logistic Regression: {'model__C': 0.03}
Random Forest: {'model__n_estimators': 100, 'model__min_samples_leaf': 6, 'model__max_features': 'sqrt', 'model__max_depth': 8}
SVM: {'model__gamma': 0.1, 'model__C': 0.3}
XGBoost: {'model__subsample': 0.8, 'model__n_estimators': 60, 'model__max_depth': 4, 'model__learning_rate': 0.03}
LightGBM: {'model__num_leaves': 15, 'model__n_estimators': 150, 'model__min_child_samples': 25, 'model__learning_rate': 0.03}


,model,cv_macro_f1,cv_macro_f1_std
2,SVM,0.4920,0.0248
0,Logistic Regression,0.4912,0.0325
1,Random Forest,0.4910,0.0272
3,XGBoost,0.4717,0.0275
4,LightGBM,0.4674,0.0373


Selected by training CV: SVM


These are tuning scores used for selection. A separate held-out evaluation is needed to estimate how the selected model generalizes on similar records.